In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("week12_transactions.csv")

conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)

query = """
SELECT *
FROM transactions
WHERE txn_type = 'send'
  AND amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5
"""

top_5 = pd.read_sql(query, conn)
print(top_5)

flagged_count = pd.read_sql(
    "SELECT COUNT(*) AS flagged_count FROM transactions WHERE flagged = 1",
    conn
)

print("\nFlagged transactions:", flagged_count.iloc[0, 0])

   txn_id        date  hour    district txn_type  amount_rwf  flagged
0  T01708  2026-11-28    15    Kicukiro     send      140000        0
1  T00364  2026-11-11    10        Huye     send      139000        0
2  T00895  2026-11-05    19     Muhanga     send      139000        0
3  T00899  2026-11-21    19  Nyarugenge     send      139000        0
4  T01314  2026-11-21    11      Gasabo     send      139000        0

Flagged transactions: 11


In [2]:
query1 = """
SELECT txn_type,
       COUNT(*) AS count,
       SUM(amount_rwf) AS total,
       AVG(amount_rwf) AS average
FROM transactions
GROUP BY txn_type
ORDER BY total DESC
"""

print(pd.read_sql(query1, conn))

query2 = """
SELECT district, SUM(amount_rwf) AS total
FROM transactions
GROUP BY district
ORDER BY total DESC
"""

print(pd.read_sql(query2, conn))

  txn_type  count     total       average
0     send    868  62049000  71485.023041
1  paybill    330  29970000  90818.181818
2  receive    489  27024000  55263.803681
3  airtime    313    833000   2661.341853
     district     total
0     Muhanga  21076000
1      Gasabo  20738000
2        Huye  18736000
3  Nyarugenge  18091000
4    Kicukiro  11338000
5     Musanze  10490500
6      Rubavu  10110500
7   Nyagatare   9296000


In [3]:
query = """
SELECT hour, COUNT(*) AS transactions
FROM transactions
GROUP BY hour
ORDER BY transactions DESC
"""

sql_result = pd.read_sql(query, conn)
print(sql_result)

pandas_result = df.groupby("hour").size().sort_values(ascending=False)

print("\nPandas result:")
print(pandas_result)

print("\nBusiest hour:", pandas_result.index[0])
print("Transactions:", pandas_result.iloc[0])

    hour  transactions
0     20           185
1     17           183
2     18           176
3     12           170
4     11           169
5     10           168
6      8           168
7     19           158
8      9           144
9     16            38
10     3            37
11    15            35
12    13            35
13    14            34
14     4            34
15     2            34
16     6            33
17     5            33
18     0            33
19     1            32
20    21            26
21    23            25
22    22            25
23     7            25

Pandas result:
hour
20    185
17    183
18    176
12    170
11    169
10    168
8     168
19    158
9     144
16     38
3      37
13     35
15     35
14     34
2      34
4      34
0      33
6      33
5      33
1      32
21     26
7      25
22     25
23     25
dtype: int64

Busiest hour: 20
Transactions: 185


In [4]:
import json

json_string = '{"city":"Kigali","temperatures":[24.5,25.1,26.3,27.0,25.8]}'

data = json.loads(json_string)

print("City:", data["city"])
print("Number of readings:", len(data["temperatures"]))
print("Average:", round(sum(data["temperatures"]) / len(data["temperatures"]), 2))

City: Kigali
Number of readings: 5
Average: 25.74


In [5]:
import json
import requests

try:
    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=-1.9441&longitude=30.0619"
        "&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali"
    )
    data = requests.get(url, timeout=10).json()

except Exception:
    with open("week12_api_sample.json") as f:
        data = json.load(f)

times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

print("Hours:", len(times))
print("Max temperature:", max(temps))
print("Min temperature:", min(temps))

hottest_index = temps.index(max(temps))
print("Hottest time:", times[hottest_index])

Hours: 24
Max temperature: 24.2
Min temperature: 16.6
Hottest time: 2026-11-23T12:00


In [6]:
weather = pd.DataFrame({
    "time": times,
    "temp_c": temps
})

print(weather.describe())

hot_hours = (weather["temp_c"] >= 25).sum()
print("Hours >= 25°C:", hot_hours)

          temp_c
count  24.000000
mean   19.466667
std     2.583504
min    16.600000
25%    17.000000
50%    19.000000
75%    21.675000
max    24.200000
Hours >= 25°C: 0


In [3]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.21.10002-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

!java -version

openjdk version "17.0.21.1" 2026-11-17
OpenJDK Runtime Environment Temurin-17.0.21.1+2 (build 17.0.21.1+2)
OpenJDK 64-Bit Server VM Temurin-17.0.21.1+2 (build 17.0.21.1+2, mixed mode, sharing)


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("week12") \
    .master("local[*]") \
    .getOrCreate()

sdf = spark.read.csv(
    "week12_transactions.csv",
    header=True,
    inferSchema=True
)

print("Row count:", sdf.count())
sdf.printSchema()
sdf.show(5)

C:\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Row count: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


In [5]:
# 3.2 Transactions above 100,000 RWF

high_value = sdf.filter(sdf.amount_rwf > 100000)

print("Transactions above 100,000 RWF:", high_value.count())

high_value.show(5)


Transactions above 100,000 RWF: 451
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00005|2026-11-15|  12|    Huye|    send|    109000|      0|
|T00009|2026-11-01|   8|  Gasabo|    send|    109000|      0|
|T00011|2026-11-22|  18|  Rubavu|    send|    102000|      0|
|T00012|2026-11-20|   9| Musanze| receive|    112000|      0|
|T00016|2026-11-03|  19| Muhanga| paybill|    154000|      0|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


In [6]:
# 3.3 Total per transaction type

type_totals = sdf.groupBy("txn_type").sum("amount_rwf")
type_totals.show()

# Total per district
district_totals = sdf.groupBy("district").sum("amount_rwf")
district_totals_pd = district_totals.toPandas()

print("District totals:")
print(district_totals_pd)

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+



C:\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Python314\Lib\site-packages\pyspark\sql\pandas\conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


District totals:
     district  sum(amount_rwf)
0    Kicukiro         11338000
1  Nyarugenge         18091000
2     Musanze         10490500
3   Nyagatare          9296000
4     Muhanga         21076000
5      Gasabo         20738000
6        Huye         18736000
7      Rubavu         10110500


In [7]:


sdf.createOrReplaceTempView("transactions")

result = spark.sql("""
SELECT txn_type, SUM(amount_rwf) AS total_amount
FROM transactions
GROUP BY txn_type
""")

result.show()

+--------+------------+
|txn_type|total_amount|
+--------+------------+
| paybill|    29970000|
| airtime|      833000|
| receive|    27024000|
|    send|    62049000|
+--------+------------+



## Part 4 — Choose Your Weapon

1. **SQL** — Because the data is already in a PostgreSQL database, and SQL is suitable for calculating monthly totals grouped by district.

2. **Pandas** — Because 40,000 rows is a manageable size for cleaning, visualization, and building a regression model.

3. **Spark** — Because 40 TB of data is too large for normal Pandas processing, while Spark is designed for large-scale distributed data.

4. **API** — Because the dashboard needs today's weather automatically without requiring a user to upload a file.